In [1]:
import math
from Utils.hdfs import init
from Utils.dataFrameSize import dfSizeEstimator
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F

In [2]:
builder = (
    SparkSession.builder
    .appName("ingest_data_from_rawfile")
    .master("yarn")
    # .config("spark.yarn.am.memory", "512m")
    # .config("spark.driver.memory", "512m")
    # .config("spark.executor.memory", "512m")
    # .config("spark.executor.cores", "1")
    # .config("spark.executor.instances", "1")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
)
spark = builder.getOrCreate()

In [3]:
init(spark)

In [4]:
%%hdfs
ls /user/jovyan/rawdata

PERMISSIONS  REPL OWNER      GROUP                 SIZE MODIFIED             NAME
------------------------------------------------------------------------------------------------------------------------
-rwxrwxr-x   1    jovyan     supergroup       622183749 2026-07-05 12:00:39.029000 flight_data.csv


In [5]:
schema = T.StructType([
    T.StructField('departure_time', T.StringType(), True),
    T.StructField('arrival_time', T.StringType(), True),
    T.StructField('flight_id', T.StringType(), True),
    T.StructField('aircraft_id', T.StringType(), True),
    T.StructField('itinerary_no', T.StringType(), True),
    T.StructField('ticket_no', T.StringType(), True),
    T.StructField('flight_cost', T.StringType(), True),
    T.StructField('origin_airport', T.StringType(), True),
    T.StructField('destination_airport', T.StringType(), True),
    T.StructField('frequent_flier', T.StringType(), True),
    T.StructField('travel_date', T.StringType(), True),
    T.StructField('airplane_model', T.StringType(), True),
    T.StructField('frequent_flier_no', T.StringType(), True),
    T.StructField('passenger_name', T.StringType(), True),
    T.StructField('passenger_country', T.StringType(), True),
    T.StructField('tail_no', T.StringType(), True),
    T.StructField('distance', T.StringType(), True),
    T.StructField('turbulance', T.StringType(), True),
    T.StructField('temp_at_dept', T.StringType(), True),
    T.StructField('fuel_consumed_litre', T.StringType(), True),
    T.StructField('taxi_duration_mins', T.StringType(), True),
    T.StructField('avg_flight_speed_kmps', T.StringType(), True),
    T.StructField('engine_performance', T.StringType(), True),
    T.StructField('passenger_dob', T.StringType(), True),
    T.StructField('passenger_flight_class', T.StringType(), True)
])

In [6]:
df = spark.read.format('csv').schema(schema).load('hdfs:///user/jovyan/rawdata/flight_data.csv')

In [7]:
df.limit(10).show()

+--------------+------------+---------+-----------+------------+---------------+-----------+--------------+--------------------+--------------+-----------+--------------------+-----------------+--------------+-----------------+-------+--------+----------+------------+-------------------+------------------+---------------------+------------------+-------------+----------------------+
|departure_time|arrival_time|flight_id|aircraft_id|itinerary_no|      ticket_no|flight_cost|origin_airport| destination_airport|frequent_flier|travel_date|      airplane_model|frequent_flier_no|passenger_name|passenger_country|tail_no|distance|turbulance|temp_at_dept|fuel_consumed_litre|taxi_duration_mins|avg_flight_speed_kmps|engine_performance|passenger_dob|passenger_flight_class|
+--------------+------------+---------+-----------+------------+---------------+-----------+--------------+--------------------+--------------+-----------+--------------------+-----------------+--------------+-----------------+-

In [8]:
%%sql
create database warehouse;

""


In [12]:
%%hdfs
ls /user

PERMISSIONS  REPL OWNER      GROUP                 SIZE MODIFIED             NAME
------------------------------------------------------------------------------------------------------------------------
drwxrwxr-x   0    jovyan     supergroup               0 2026-07-05 12:04:08.889000 jovyan


In [9]:
%%sql
use warehouse;

""


In [10]:
df.repartition(max(
    3, #number of executors
    math.ceil(dfSizeEstimator(df, True)) / 200)).write.mode('overwrite').format('delta')\
    .option("path", "hdfs:///user/jovyan/lakehouse/bronze/flight")\
    .saveAsTable('warehouse.flight_raw')

In [11]:
df.rdd.getNumPartitions()

5

In [12]:
%%sql
select *
from delta.`hdfs:///user/jovyan/lakehouse/bronze/flight` limit 3;

,departure_time,arrival_time,flight_id,aircraft_id,itinerary_no,ticket_no,flight_cost,origin_airport,destination_airport,frequent_flier,...,tail_no,distance,turbulance,temp_at_dept,fuel_consumed_litre,taxi_duration_mins,avg_flight_speed_kmps,engine_performance,passenger_dob,passenger_flight_class
0,22:08:41,23:57:41,CCK50278,AI947,61143910,SS-34573-UW-271,3940.67,Cape Coral,Paducah,True,...,S873RN,1724,0,-7.5,24177.5,25.0,834.3,12773,1999-11-08,economy
1,18:29:30,19:17:30,BRT38419,AI438,68917462,NT-33366-BW-995,698.1,Carrboro,Hilliard,True,...,H544XD,5842,0,-7.3,59102.8,120.0,1127.3,46453,2001-01-27,economy
2,13:32:41,15:14:41,KEU89486,AI045,62198786,BR-03300-EE-686,784.61,North Richland Hills,Central Falls,False,...,Z674GW,2008,1,-9.7,19477.8,84.0,945.9,18050,2003-03-13,economy


In [13]:
%%sql
select * from (
select row_number() over (order by add) as rn, a.* 
from json.`hdfs:///user/jovyan/lakehouse/bronze/flight/_delta_log/00000000000000000000.json` a)
where rn = 4;

Stored Spark DataFrame in 'output'


In [14]:
output = output.toPandas().to_numpy().tolist()[0][1]

In [15]:
import ast

In [16]:
output = ast.literal_eval(output.asDict()['stats'])

In [17]:
output

{'numRecords': 1000001,
 'minValues': {'departure_time': '00:00:00',
  'arrival_time': '00:00:00',
  'flight_id': 'AAA00237',
  'aircraft_id': 'AI000',
  'itinerary_no': '60000005',
  'ticket_no': 'AA-00019-TA-797',
  'flight_cost': '1000.0',
  'origin_airport': 'Aberdeen',
  'destination_airport': 'Aberdeen',
  'frequent_flier': 'False',
  'travel_date': '2024-05-13',
  'airplane_model': '"Airbus A300-600ST ""Beluga"""',
  'frequent_flier_no': '000001YV',
  'passenger_name': 'Aaron',
  'passenger_country': 'Afghanistan',
  'tail_no': 'A000AJ',
  'distance': '1000',
  'turbulance': '0',
  'temp_at_dept': '-0.0',
  'fuel_consumed_litre': '12000.0',
  'taxi_duration_mins': '100.0',
  'avg_flight_speed_kmps': '1000.0',
  'engine_performance': '10000',
  'passenger_dob': '1998-01-01',
  'passenger_flight_class': 'economy'},
 'maxValues': {'departure_time': '23:59:59',
  'arrival_time': '23:59:59',
  'flight_id': 'ZZZ98392',
  'aircraft_id': 'AI999',
  'itinerary_no': '69999996',
  'ticket_